In [10]:
import tensorflow as tf
import os
import pathlib

# Download the official tensorflow flower archive
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, extract=True)
data_dir = pathlib.Path(data_dir).parent / 'flower_photos'

print(f"✅ Raw photos successfully downloaded to: {data_dir}")

✅ Raw photos successfully downloaded to: /root/.keras/datasets/flower_photos


In [14]:
import tensorflow as tf
import os
import tarfile

# 1. Force download and clean extraction to guarantee folder structure
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
local_tgz = tf.keras.utils.get_file('flower_photos.tgz', origin=url)

# Extract it manually to ensure classes are subdirectories
extract_path = "/content/fresh_dataset"
if not os.path.exists(extract_path):
    with tarfile.open(local_tgz, "r:gz") as tar:
        tar.extractall(path=extract_path)

# Correct source directory where class subfolders live
source_dir = os.path.join(extract_path, "flower_photos")

# 2. Pipeline Configuration
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("--- Processing & Creating Training Dataset (70%) ---")
train_ds = tf.keras.utils.image_dataset_from_directory(
    source_dir,
    validation_split=0.30,   # Reserve 30% total for validation/testing combined
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

print("\n--- Processing Remaining Data for Val & Test Splits ---")
remaining_ds = tf.keras.utils.image_dataset_from_directory(
    source_dir,
    validation_split=0.30,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Split the remaining 30% into equal 15% validation and 15% testing sets [cite: 7]
cardinality = tf.data.experimental.cardinality(remaining_ds).numpy()
val_batches = cardinality // 2

val_ds = remaining_ds.take(val_batches)
test_ds = remaining_ds.skip(val_batches)

# Optimize memory pipeline configurations
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print("\n📊 Final Verification Check:")
print(f"✅ Training batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"✅ Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"✅ Testing batches: {tf.data.experimental.cardinality(test_ds).numpy()}")
print("\n🚀 DATASET PREPARATION COMPLETE! Ready to hand off to your team.")

228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/tmp/ipykernel_9808/1090567830.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


--- Processing & Creating Training Dataset (70%) ---
Found 3670 files belonging to 5 classes.
Using 2569 files for training.

--- Processing Remaining Data for Val & Test Splits ---
Found 3670 files belonging to 5 classes.
Using 1101 files for validation.

📊 Final Verification Check:
✅ Training batches: 81
✅ Validation batches: 17
✅ Testing batches: 18

🚀 DATASET PREPARATION COMPLETE! Ready to hand off to your team.
